> **Original course-project notebook (2026), kept for reference; superseded by `src/dfer/`.** Outputs are cleared: they contained DFEW frames (licence-restricted) and results that were selected on the fold's test set and affected by a frame-ordering bug. See the README for the corrected pipeline and results.

#Self-Supervised VideoMAE for DFER

In this notebook, we implement a self-supervised version of VideoMAE pretrained on Kinetics-400. The goal is to fine-tune the model on the DFEW dataset for emotion classification.

We perform the following steps:

* Load and prepare the DFEW dataset.
* Apply transformations to train and validation data.
* Use the VideoMAE pretrained model for fine-tuning.
* Calculate class weights to handle class imbalance.

This setup allows us to evaluate how self-supervised pretraining on a large dataset like Kinetics-400 can improve performance on the smaller DFEW dataset.

##Imports and Setup

We begin by importing the necessary libraries for dataset processing, model training, and evaluation:

* PyTorch: For building and training the model.
* Transformers: For loading the pretrained VideoMAE model.
* matplotlib and tqdm: For visualization and progress bars.
* yaml: For loading configuration files.

We also define paths for the project and ensure that the source directory is included in the system path.

In [ ]:
# =========================
# 0) Imports + Paths
# =========================
import os, sys, time, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

from tqdm import tqdm
import yaml

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

CONFIG_PATH = PROJECT_ROOT / "configs" / "paths.yaml"

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

DFEW_ROOT = config["dfew_root"]
SPLITS_ROOT = config["splits_root"]

OUTPUT_DIR = PROJECT_ROOT / "outputs"
CKPT_DIR   = OUTPUT_DIR / "checkpoints"
LOG_DIR    = OUTPUT_DIR / "logs"
PLOT_DIR   = OUTPUT_DIR / "plots"

for d in [OUTPUT_DIR, CKPT_DIR, LOG_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("DFEW_ROOT:", DFEW_ROOT)

##Reproducibility and Device Setup

In [ ]:
# =========================
# 1) Reproducibility + Device
# =========================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

##Dataset Paths and Initialization

In [ ]:
FOLD = 1

TRAIN_CSV = os.path.join(SPLITS_ROOT, "train(single-labeled)", f"set_{FOLD}.csv")
VAL_CSV   = os.path.join(SPLITS_ROOT, "test(single-labeled)",  f"set_{FOLD}.csv")

RUN_NAME = f"mae_ssl_fold{FOLD}_FINAL"

##Data Loading and Transformation

In this section, we initialize the train and validation datasets using the DFEWVideoDataset class. We define two sets of transformations:

* Training transformations: Random resizing, horizontal flipping, and normalization to augment the dataset.
* Validation transformations: Resizing and normalization to prepare the data for evaluation.

We also set up DataLoaders for both the training and validation sets, which will handle batching and shuffling during training.

In [ ]:
from dataset import DFEWVideoDataset

NUM_CLASSES = 7
NUM_FRAMES = 16
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485,0.456,0.406),
                         std=(0.229,0.224,0.225))
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485,0.456,0.406),
                         std=(0.229,0.224,0.225))
])

train_ds = DFEWVideoDataset(DFEW_ROOT, TRAIN_CSV, train_transform, NUM_FRAMES)
val_ds   = DFEWVideoDataset(DFEW_ROOT, VAL_CSV,   val_transform,   NUM_FRAMES)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=4, pin_memory=True)

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))

##Class Distribution and Class Weights

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
train_counts = train_df["label"].value_counts().sort_index()

freq = train_counts.values.astype(np.float32)
weights = 1.0 / (freq + 1e-6)
weights = weights / weights.sum() * len(weights)

class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)

##Supervised VideoMAE Model Setup

In this section, we load the pretrained VideoMAE model that has been fine-tuned on Kinetics-400. The model is used as a backbone for the emotion classification task on the DFEW dataset. We set the following hyperparameters for training:

* EPOCHS: Number of training epochs.
* Learning Rate (LR): Set to 3e-5 to allow fine-tuning without overfitting.
* Weight Decay: To prevent overfitting, we use 0.05 as the weight decay value.
* Loss function: We use weighted cross-entropy loss to address class imbalance in the DFEW dataset.

In [ ]:
from transformers import VideoMAEForVideoClassification

EPOCHS = 20
LR = 1e-5
WEIGHT_DECAY = 0.05
PATIENCE = 5
MIN_DELTA = 1e-4

model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base",  # SELF-SUPERVISED
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

scaler = torch.amp.GradScaler("cuda") if device=="cuda" else None

print("MAE SSL backbone loaded.")

##Checkpoint and Resume Utilities

This section defines the paths for saving and loading model checkpoints during training. It also includes functionality to resume training from the last saved checkpoint if needed. The following files are defined:

* Best accuracy (best_acc) and best UAR (best_uar) model weights.
* Last model checkpoint to resume training from the last epoch.
* History of training progress (loss, accuracy, UAR) is saved for future reference.
* Confusion matrix plots for both best accuracy and best UAR.

These utilities ensure that the model's training progress is saved periodically and can be resumed if interrupted.

In [ ]:
BEST_ACC_PATH = CKPT_DIR / f"{RUN_NAME}_best_acc.pth"
BEST_UAR_PATH = CKPT_DIR / f"{RUN_NAME}_best_uar.pth"
LAST_PATH     = CKPT_DIR / f"{RUN_NAME}_last.pth"

HISTORY_CSV = LOG_DIR / f"{RUN_NAME}_history.csv"

##Training and Validation Loops

In [ ]:
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "val_uar": [],
    "lr": []
}

best_val_acc = -1.0
best_val_uar = -1.0
no_improve_uar = 0

for epoch in range(EPOCHS):

    model.train()
    train_losses, train_preds, train_targets = [], [], []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for videos, labels in pbar:

        if videos.shape[1] == 3:
            videos = videos.permute(0,2,1,3,4)

        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        if device=="cuda":
            with torch.amp.autocast("cuda"):
                logits = model(videos).logits
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(videos).logits
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        train_losses.append(loss.item())
        train_preds.extend(logits.argmax(1).cpu().numpy())
        train_targets.extend(labels.cpu().numpy())

    train_loss = np.mean(train_losses)
    train_acc = accuracy_score(train_targets, train_preds)

    # -------- VALIDATION --------
    model.eval()
    val_losses, val_preds, val_targets = [], [], []

    with torch.no_grad():
        for videos, labels in val_loader:

            if videos.shape[1] == 3:
                videos = videos.permute(0,2,1,3,4)

            videos = videos.to(device)
            labels = labels.to(device)

            logits = model(videos).logits
            loss = criterion(logits, labels)

            val_losses.append(loss.item())
            val_preds.extend(logits.argmax(1).cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    val_loss = np.mean(val_losses)
    val_acc  = accuracy_score(val_targets, val_preds)
    val_uar  = recall_score(val_targets, val_preds, average="macro")

    scheduler.step()

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss {train_loss:.4f} | Acc {train_acc:.4f}")
    print(f"Val   Loss {val_loss:.4f} | Acc {val_acc:.4f} | UAR {val_uar:.4f}")

    history["epoch"].append(epoch+1)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_uar"].append(val_uar)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)

    # BEST ACC
    if val_acc > best_val_acc + MIN_DELTA:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_ACC_PATH)
        print("New BEST-ACC saved.")

    # BEST UAR (PRIMARY)
    if val_uar > best_val_uar + MIN_DELTA:
        best_val_uar = val_uar
        torch.save(model.state_dict(), BEST_UAR_PATH)
        no_improve_uar = 0
        print("New BEST-UAR saved.")
    else:
        no_improve_uar += 1
        print(f"No UAR improvement {no_improve_uar}/{PATIENCE}")

    if no_improve_uar >= PATIENCE:
        print("Early stopping (UAR).")
        break

print("Training finished.")
print("Best Acc:", best_val_acc)
print("Best UAR:", best_val_uar)

##PLots (acc , loss , UAR)

In [ ]:
hist = pd.read_csv(HISTORY_CSV)

plt.figure()
plt.plot(hist["epoch"], hist["train_loss"], label="train_loss")
plt.plot(hist["epoch"], hist["val_loss"], label="val_loss")
plt.legend(); plt.grid()
plt.title(RUN_NAME + " Loss")
plt.savefig(PLOT_DIR / f"{RUN_NAME}_loss.png", dpi=200)
plt.show()

plt.figure()
plt.plot(hist["epoch"], hist["train_acc"], label="train_acc")
plt.plot(hist["epoch"], hist["val_acc"], label="val_acc")
plt.legend(); plt.grid()
plt.title(RUN_NAME + " Accuracy")
plt.savefig(PLOT_DIR / f"{RUN_NAME}_accuracy.png", dpi=200)
plt.show()

plt.figure()
plt.plot(hist["epoch"], hist["val_uar"], label="val_uar")
plt.legend(); plt.grid()
plt.title(RUN_NAME + " UAR")
plt.savefig(PLOT_DIR / f"{RUN_NAME}_uar.png", dpi=200)
plt.show()

##Confusion Matrices (acc , UAR)

In [ ]:
def eval_cm(weights_path, title, save_path):
    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.to(device)
    model.eval()

    all_preds, all_targets = [], []

    with torch.no_grad():
        for videos, labels in tqdm(val_loader, leave=False):
            if videos.shape[1] == 3:
                videos = videos.permute(0,2,1,3,4)
            videos = videos.to(device)
            logits = model(videos).logits
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_targets.extend(labels.numpy())

    cm = confusion_matrix(all_targets, all_preds)

    plt.figure(figsize=(6,6))
    plt.imshow(cm)
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()

    print(title,
          "Acc:", accuracy_score(all_targets, all_preds),
          "UAR:", recall_score(all_targets, all_preds, average="macro"))

eval_cm(BEST_ACC_PATH,
        f"{RUN_NAME} BEST ACC",
        PLOT_DIR / f"{RUN_NAME}_cm_best_acc.png")

eval_cm(BEST_UAR_PATH,
        f"{RUN_NAME} BEST UAR",
        PLOT_DIR / f"{RUN_NAME}_cm_best_uar.png")